# CUSP Quickstart

**CUSP (Cutoff-conditioned Unseen Scientific Progress)** — a benchmark for evaluating whether AI systems can forecast scientific discoveries made after their training cutoff.

- **Dataset:** https://huggingface.co/datasets/SeanWu25/CUSP  
- **Code:** https://github.com/SeanWu25/cusp-scientific-foresight

This notebook walks through the full pipeline:
1. Load the dataset from HuggingFace
2. Explore benchmark items
3. Run a model to generate predictions (`test_LLM.py`)
4. Evaluate predictions with the LLM judge (`eval.py`)
5. (Optional) Re-run with You.com web search (`web_search.py`)

In [ ]:
# !pip install huggingface_hub openai anthropic python-dotenv tqdm pandas

In [ ]:
import sys, json, os
sys.path.insert(0, '../scripts')   # so we can import eval, test_LLM, web_search

from huggingface_hub import hf_hub_download

---
## 1. Load the Dataset

In [ ]:
dataset_path = hf_hub_download(
    repo_id="SeanWu25/CUSP",
    filename="CUSP_final.jsonl",
    repo_type="dataset",
)

with open(dataset_path) as f:
    records = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(records)} items")
print("Fields:", list(records[0].keys()))

---
## 2. Explore

In [ ]:
import pandas as pd
df = pd.DataFrame(records)

print("By source:");  print(df['source'].value_counts().to_string())
print("\nBy domain:"); print(df['main_area'].value_counts().to_string())

In [ ]:
# Inspect a single item
item = next(r for r in records
            if r.get('binary_question') and r.get('mcq_question') and r.get('frq_prompt'))

print(f"Source:    {item['source']}  |  Domain: {item.get('main_area')}")
print(f"Published: {item['publication_date']}  |  Cutoff: {item.get('cutoff_date','N/A')}")
print(f"\nAbstract:\n{str(item.get('abstract',''))[:300]}...\n")
print(f"[Binary]    {item['binary_question']}  → {item['binary_answer_key']}")
print(f"[Perturbed] {item['binary_question_perturbed']}  → {item['binary_perturbed_answer_key']}")
print(f"[MCQ]       {item['mcq_question']}")
for c in item.get('mcq_choices', []): print(f"            {c}")
print(f"            → {item['mcq_answer_key']}")
print(f"[FRQ]       {item['frq_prompt']}")
print(f"[Date]      {item['date_prediction_prompt']}  → {item['ground_truth_date']}")

---
## 3. Run Model Inference

Set your API credentials — either in a `.env` file or directly below.

In [ ]:
# Option A: load from .env file
from dotenv import load_dotenv
load_dotenv('../.env')

# Option B: set directly
# os.environ['OPENAI_API_KEY'] = 'sk-...'
# os.environ['AZURE_OPENAI_ENDPOINT'] = 'https://...'
# os.environ['AZURE_OPENAI_KEY'] = '...'
# os.environ['AZURE_OPENAI_DEPLOYMENT'] = 'gpt-4o'

In [ ]:
from test_LLM import run_inference

predictions_path = run_inference(
    output_path      = 'predictions.jsonl',
    provider         = 'OPENAI',        # or 'AZURE', 'ANTHROPIC'
    model            = 'gpt-4o',
    knowledge_cutoff = '2023-10',       # GPT-4o training cutoff: October 2023
    max_rows         = 20,              # remove for full run
    verbose          = True,
)

print(f"\nPredictions saved to: {predictions_path}")

---
## 4. Evaluate Predictions

Runs both Track 1 (outcome: binary, MCQ, FRQ, date) and Track 2 (reasoning: leakage, mechanistic, constraint) using an LLM judge with web search.

In [ ]:
from eval import run_evaluation

report = run_evaluation(
    predictions_path = predictions_path,
    model            = 'gpt-5.4-mini',  # judge model
    output_path      = 'report.json',
    verbose          = True,
)

# ── Per-task metrics ──────────────────────────────────────────────────────────
print("\n=== Task Metrics ===")
for task, m in report['task_metrics'].items():
    if task == 'frq':
        print(f"  frq:                   score={m.get('mean_score','?'):.3f}  (n={m['count']})")
        for sub in ('alignment', 'specificity', 'novelty', 'feasibility'):
            val = m.get(f'mean_{sub}')
            if val is not None:
                print(f"    {sub:<16} {val:.3f}")
    elif 'accuracy' in m:
        print(f"  {task:<22} accuracy={m['accuracy']:.1%}  (n={m['count']})")
    elif 'mean_score' in m:
        print(f"  {task:<22} score={m['mean_score']:.3f}  (n={m['count']})")

# ── Leakage judge ─────────────────────────────────────────────────────────────
lm = report.get('reasoning_metrics', {}).get('leakage', {})
if lm:
    print(f"\n=== Leakage Judge ===")
    print(f"  pass rate: {lm.get('pass_rate', 0):.1%}  (n={lm.get('count', 0)})")

---
## 5. (Optional) Web Search Ablation

Re-run inference with You.com web search enabled, then compare scores.  
Requires a `YOU_API_KEY` — get one at https://you.com/api

In [ ]:
os.environ['YOU_API_KEY'] = 'ydc-...'  # set your You.com API key here

predictions_ws_path = run_inference(
    output_path      = 'predictions_websearch.jsonl',
    provider         = 'OPENAI',
    model            = 'gpt-4o',
    knowledge_cutoff = '2023-10',
    max_rows         = 20,
    web_search       = True,
    ws_cutoff        = '2023-10',
    you_api_key      = os.environ.get('YOU_API_KEY'),
)

print(f"Web search predictions saved to: {predictions_ws_path}")

report_ws = run_evaluation(
    predictions_path = predictions_ws_path,
    model            = 'gpt-5.4-mini',
    output_path      = 'report_websearch.json',
    verbose          = True,
)

print("\n=== Without web search ===")
for task, m in report['task_metrics'].items():
    if 'accuracy' in m:
        print(f"  {task:<22} accuracy={m['accuracy']:.1%}")

print("\n=== With web search ===")
for task, m in report_ws['task_metrics'].items():
    if 'accuracy' in m:
        print(f"  {task:<22} accuracy={m['accuracy']:.1%}")

---
## 6. Inspect Individual Results

In [ ]:
for row in report['results'][:3]:
    print(f"id={row['id']}")
    for task, res in row.get('tasks', {}).items():
        if not res.get('skipped'):
            if task == 'frq':
                print(f"  frq: score={res.get('score')}  alignment={res.get('alignment')}  "
                      f"specificity={res.get('specificity')}  novelty={res.get('novelty')}")
            elif task == 'date':
                print(f"  date: exact={res.get('exact_match')}  distance={res.get('month_distance')}mo")
            else:
                print(f"  {task}: correct={res.get('correct')}  parsed={res.get('parsed_answer')}  gt={res.get('ground_truth')}")
    leak = row.get('reasoning', {}).get('leakage', {})
    if leak:
        print(f"  leakage: verdict={leak.get('verdict')}  score={leak.get('score')}")
    print()
